# OpenWebText Scale-Up: Fock-PARFLM v2.1 (v_hidden adaptive)

## Motivation

Train the Fock-PARFLM v2.1 branch on OpenWebText using the same protocol as
the Phase 3 Multi-Xi SPLM scale-up, completing the three-branch comparison
(SPLM vs PARFLM vs Fock-PARFLM) on realistic text.

Fock-PARFLM v2.1 extends Multi-Xi PARFLM with M=16 latent register particles
using Q/K/V-structured creation gates, LIFO stack discipline, and a reverse
channel (non-conservative exchange force). This is the best TinyStories model
(9.30 PPL) and should show how Fock registers behave on richer data.

## Config summary

| Parameter | Phase 3 SPLM | PARFLM | **This notebook (Fock)** |
|-----------|-------------|--------|-------------------------|
| Model | Multi-Xi SPLM | Multi-Xi PARFLM | **Fock-PARFLM v2.1** |
| `K` (xi channels) | 4 | 8 | **4** |
| `H` (V_phi hidden) | -- | 128 | **128** |
| `M` (Fock registers) | -- | -- | **16** |
| `d_k` | -- | -- | **64** |
| Fock version | -- | -- | **v2 (B1+B2+B3)** |
| Reverse channel | -- | -- | **True** |
| Stack discipline | -- | -- | **LIFO** |
| `v_hidden` | 2048 | adaptive | **adaptive** |
| `v_depth` | 4 | adaptive | **adaptive** |
| Steps | 100,000 | 100,000 | **100,000** |

In [ ]:
# ── Cell 1: Environment + Drive mount ─────────────────────────────
import subprocess, sys, os, gc, math, json, time
from pathlib import Path
from dataclasses import asdict, fields as dc_fields

def run(cmd):
    print(f"$ {cmd}")
    subprocess.check_call(cmd, shell=True)

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    run('pip install -q transformers huggingface_hub pyarrow datasets')
    if not os.path.isdir('semsimula-paper'):
        run('git clone --depth 1 https://github.com/dimitarpg13/semsimula-paper.git')
    REPO = 'semsimula-paper'
else:
    REPO = os.environ.get('SEMSIMULA_PAPER', '.')

ARCH_DIR = os.path.join(REPO, 'notebooks', 'conservative_arch')
for p in [
    ARCH_DIR,
    os.path.join(ARCH_DIR, 'multixi'),
    os.path.join(ARCH_DIR, 'parf'),
    os.path.join(ARCH_DIR, 'energetic_minima'),
    os.path.join(ARCH_DIR, 'sarf_mass_variant'),
    os.path.join(ARCH_DIR, 'scaleup'),
]:
    if p not in sys.path:
        sys.path.insert(0, p)

import torch
import torch.nn as nn
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False

DEVICE = (
    'cuda' if torch.cuda.is_available()
    else ('mps' if hasattr(torch.backends, 'mps')
          and torch.backends.mps.is_available() else 'cpu')
)
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    GPU_NAME = torch.cuda.get_device_name()
    VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {GPU_NAME}  VRAM: {VRAM_GB:.1f} GB')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/semsimula_fock_multixi_openwebtext_scaleup')
else:
    DRIVE_ROOT = Path.home() / 'semsimula_fock_multixi_openwebtext_scaleup'

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
CKPT_DIR = DRIVE_ROOT / 'checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = DRIVE_ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = DRIVE_ROOT / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f'Drive root   : {DRIVE_ROOT}')
print(f'Checkpoints  : {CKPT_DIR}')
print(f'Results      : {RESULTS_DIR}')

In [ ]:
# ── Cell 2: Checkpoint resolution + resume detection ─────────────
CKPT_STEPS = [25_000, 50_000, 75_000, 100_000]
CKPT_PREFIX = 'fock_multixi_owt_scaleup'

resume_step = 0
resume_ckpt = None
for s in sorted(CKPT_STEPS, reverse=True):
    cand = CKPT_DIR / f'{CKPT_PREFIX}_step{s}.pt'
    if cand.exists():
        resume_ckpt = cand
        resume_step = s
        break

if resume_ckpt is not None:
    print(f'Found checkpoint at step {resume_step}: {resume_ckpt}')
    print(f'Training will resume from step {resume_step + 1}.')
else:
    print('No checkpoint found — training from scratch.')

In [ ]:
# ── Cell 3: Config ────────────────────────────────────────────────
# Training protocol (matches Phase 3 SPLM and PARFLM notebooks)
TOTAL_STEPS   = 100_000
LR            = 5e-4
WEIGHT_DECAY  = 0.01
WARMUP_STEPS  = 2000
GRAD_CLIP     = 1.0
FOCK_GRAD_CLIP = 0.5
EVAL_INTERVAL = 1000
EVAL_ITERS    = 40
LOG_INTERVAL  = 100
BLOCK_SIZE    = 512
SEED          = 0

# Multi-Xi
XI_CHANNELS        = 4
XI_ALPHA_INITS     = [0.0, 0.5, 0.9, 0.99]
XI_ALPHA_INIT_MODE = 'explicit'
XI_TAU_MAX         = 100.0
FIXED_GAMMA        = 0.30

# PARF pair forces
V_PHI_KIND         = 'structural_competitive'
V_PHI_H            = 128
V_PHI_D_TYPE       = 32
V_PHI_D_ANGLE      = 16
V_PHI_MLP_HIDDEN   = 32
TOP_K              = 8
SCORE_HEAD_HIDDEN  = 32
GUMBEL_TAU_INIT    = 1.0
GUMBEL_TAU_MIN     = 0.3

# Fock-specific (v2.1 = v2 + B1 + B2 + B3)
FOCK_VERSION       = 'v2'
N_REGISTERS        = 16
D_K                = 64
STACK_DISCIPLINE   = True
REVERSE_CHANNEL    = True
REGISTER_SALIENCE_DECAY     = 0.5
REGISTER_SALIENCE_THRESHOLD = 0.005
TAU_CREATE_INIT    = 8.0
PER_REGISTER_TAU   = True
PER_REGISTER_KEYS  = True
ORTHO_REGISTER_INIT = True

# Adaptive V_theta targets
V_HIDDEN_TARGET = 2048
V_DEPTH_TARGET  = 4
V_HIDDEN_FALLBACK = 1024
V_DEPTH_FALLBACK  = 3

# Data
MAX_TRAIN_TOKENS = 200_000_000
VAL_TOKENS       = 2_000_000
VOCAB_SIZE       = 50257

print(f'Config: K={XI_CHANNELS}  H={V_PHI_H}  top_k={TOP_K}  M={N_REGISTERS}')
print(f'  Fock {FOCK_VERSION}  LIFO={STACK_DISCIPLINE}  reverse={REVERSE_CHANNEL}')
print(f'  B1(per_register_tau)={PER_REGISTER_TAU}  B2(per_register_keys)={PER_REGISTER_KEYS}  B3(ortho_init)={ORTHO_REGISTER_INIT}')
print(f'  v_hidden={V_HIDDEN_TARGET}/{V_HIDDEN_FALLBACK}  v_depth={V_DEPTH_TARGET}/{V_DEPTH_FALLBACK}')
print(f'  gumbel_tau: {GUMBEL_TAU_INIT} -> {GUMBEL_TAU_MIN}')
print(f'Steps: {TOTAL_STEPS:,}  LR: {LR}  Warmup: {WARMUP_STEPS}')

In [ ]:
# ── Cell 4: Data loading (reuse cached data) ─────────────────────
CHUNK_SIZE = 50_000
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained('gpt2')

train_cache = DATA_DIR / f'openwebtext_train_{MAX_TRAIN_TOKENS // 1_000_000}M.npy'
val_cache   = DATA_DIR / f'openwebtext_val_{VAL_TOKENS // 1_000_000}M.npy'

# Check existing data caches (SPLM Phase 3, SPLM Phase 2, PARFLM)
for alt_name in [
    'semsimula_splm_openwebtext_scaleup',
    'semsimula_parf_multixi_openwebtext_scaleup',
    'semsimula_splm_openwebtext',
]:
    if train_cache.exists():
        break
    if IN_COLAB:
        alt = Path(f'/content/drive/MyDrive/{alt_name}/data')
    else:
        alt = Path.home() / alt_name / 'data'
    alt_train = alt / f'openwebtext_train_{MAX_TRAIN_TOKENS // 1_000_000}M.npy'
    alt_val   = alt / f'openwebtext_val_{VAL_TOKENS // 1_000_000}M.npy'
    if alt_train.exists():
        import shutil
        print(f'Reusing data cache from {alt}')
        shutil.copy2(str(alt_train), str(train_cache))
        shutil.copy2(str(alt_val), str(val_cache))
        break

if train_cache.exists() and val_cache.exists():
    print('Loading cached OpenWebText tokens...')
    train_ids = np.load(str(train_cache))
    val_ids   = np.load(str(val_cache))
    print(f'  train: {len(train_ids):,} tokens')
    print(f'  val:   {len(val_ids):,} tokens')
else:
    from datasets import load_dataset
    print(f'Streaming OpenWebText (target: {MAX_TRAIN_TOKENS:,} train + {VAL_TOKENS:,} val tokens)...')
    ds = load_dataset('Skylion007/openwebtext', split='train', streaming=True, trust_remote_code=True)
    all_ids = []
    total = 0
    target = MAX_TRAIN_TOKENS + VAL_TOKENS
    chunk_texts = []
    n_docs = 0
    t0 = time.time()
    for example in ds:
        chunk_texts.append(example['text'])
        n_docs += 1
        if len(chunk_texts) >= CHUNK_SIZE:
            joined = '\n\n'.join(chunk_texts)
            chunk_ids = tok.encode(joined)
            all_ids.extend(chunk_ids)
            total = len(all_ids)
            elapsed = time.time() - t0
            print(f'  {n_docs:,} docs  {total:,} tokens  ({elapsed:.0f}s)', flush=True)
            chunk_texts = []
            del joined, chunk_ids
            if total >= target:
                break
    if chunk_texts:
        joined = '\n\n'.join(chunk_texts)
        all_ids.extend(tok.encode(joined))
        del joined, chunk_texts
    all_ids = np.array(all_ids, dtype=np.uint16)
    total = len(all_ids)
    print(f'Total streamed: {total:,} tokens from {n_docs:,} documents ({time.time() - t0:.0f}s)')
    val_ids   = all_ids[-VAL_TOKENS:]
    train_ids = all_ids[:-VAL_TOKENS]
    if len(train_ids) > MAX_TRAIN_TOKENS:
        train_ids = train_ids[:MAX_TRAIN_TOKENS]
    del all_ids
    np.save(str(train_cache), train_ids)
    np.save(str(val_cache), val_ids)
    print(f'  Cached: train={len(train_ids):,} -> {train_cache}')
    print(f'  Cached: val={len(val_ids):,}   -> {val_cache}')

# ── Logfreq surprisal ──
LOGFREQ_PATH = str(DATA_DIR / 'logfreq_surprisal_openwebtext.npy')
for alt_name in [
    'semsimula_splm_openwebtext_scaleup',
    'semsimula_parf_multixi_openwebtext_scaleup',
    'semsimula_splm_openwebtext',
]:
    if os.path.exists(LOGFREQ_PATH):
        break
    if IN_COLAB:
        alt_lf = Path(f'/content/drive/MyDrive/{alt_name}/data/logfreq_surprisal_openwebtext.npy')
    else:
        alt_lf = Path.home() / alt_name / 'data' / 'logfreq_surprisal_openwebtext.npy'
    if alt_lf.exists():
        import shutil
        print(f'Reusing logfreq from {alt_lf}')
        shutil.copy2(str(alt_lf), LOGFREQ_PATH)
        break

if not os.path.exists(LOGFREQ_PATH):
    print('Computing unigram surprisal from OpenWebText training tokens...')
    counts = np.bincount(train_ids, minlength=VOCAB_SIZE).astype(np.int64)
    N = len(train_ids)
    p = (counts + 1.0) / (N + VOCAB_SIZE)
    surprisal = -np.log(p).astype(np.float32)
    print(f'  Surprisal: min={surprisal.min():.3f}  max={surprisal.max():.3f}  '
          f'mean={surprisal.mean():.3f}')
    np.save(LOGFREQ_PATH, surprisal)
    del counts, p, surprisal

gc.collect()
print(f'\nReady: train={len(train_ids):,}  val={len(val_ids):,}')
print(f'Logfreq: {LOGFREQ_PATH}')

In [ ]:
# ── Cell 5: Model init with adaptive V_theta sizing ───────────────
from model_fock_parf_multixi import FockMultiXiPARFConfig, FockMultiXiPARFLM
from data_module import get_batch

def make_config(v_hidden, v_depth):
    return FockMultiXiPARFConfig(
        d=256,
        max_len=1024,
        v_hidden=v_hidden,
        v_depth=v_depth,
        L=12,
        init_m=1.0,
        init_gamma=1.0,
        vocab_size=VOCAB_SIZE,
        mass_mode='logfreq',
        logfreq_init_alpha=0.1,
        logfreq_path=LOGFREQ_PATH,
        ln_after_step=True,
        fixed_gamma=FIXED_GAMMA,
        causal_force=True,
        # Multi-Xi
        xi_channels=XI_CHANNELS,
        xi_alpha_inits=XI_ALPHA_INITS,
        xi_learnable=True,
        xi_alpha_init_mode=XI_ALPHA_INIT_MODE,
        xi_tau_max=XI_TAU_MAX,
        # PARF pair forces
        v_phi_kind=V_PHI_KIND,
        v_phi_phi_hidden=V_PHI_H,
        v_phi_theta_hidden=V_PHI_H,
        v_phi_d_type=V_PHI_D_TYPE,
        v_phi_d_angle=V_PHI_D_ANGLE,
        v_phi_mlp_hidden=V_PHI_MLP_HIDDEN,
        top_k=TOP_K,
        score_head_hidden=SCORE_HEAD_HIDDEN,
        gumbel_tau_init=GUMBEL_TAU_INIT,
        gumbel_tau_min=GUMBEL_TAU_MIN,
        # Memory optimisations
        use_gathered_v_phi=True,
        use_layer_checkpoint=True,
        ln_before_distance=True,
        per_layer_v_phi_scale=True,
        # Fock-specific
        fock_version=FOCK_VERSION,
        n_registers=N_REGISTERS,
        d_k=D_K,
        stack_discipline=STACK_DISCIPLINE,
        reverse_channel=REVERSE_CHANNEL,
        register_salience_decay=REGISTER_SALIENCE_DECAY,
        register_salience_threshold=REGISTER_SALIENCE_THRESHOLD,
        tau_create_init=TAU_CREATE_INIT,
        per_register_tau=PER_REGISTER_TAU,
        per_register_keys=PER_REGISTER_KEYS,
        ortho_register_init=ORTHO_REGISTER_INIT,
    )

def try_model(v_hidden, v_depth):
    """Instantiate model and run OOM probe. Returns (model, cfg) or raises."""
    cfg = make_config(v_hidden, v_depth)
    mdl = FockMultiXiPARFLM(cfg).to(DEVICE)
    n = mdl.num_params()
    print(f'  Trying v_hidden={v_hidden} v_depth={v_depth} -> {n:,} params')
    if DEVICE == 'cuda':
        _rng = np.random.default_rng(42)
        _xb, _yb = get_batch(train_ids, 2, BLOCK_SIZE, _rng)
        _x = torch.from_numpy(_xb).to(DEVICE)
        _y = torch.from_numpy(_yb).to(DEVICE)
        _, _loss = mdl(_x, _y)
        _loss.backward()
        mdl.zero_grad(set_to_none=True)
        del _x, _y, _xb, _yb, _loss
        torch.cuda.empty_cache()
        print(f'  OOM probe passed (batch=2)')
    return mdl, cfg

# ── Adaptive sizing: try target, fall back ──
model = None
model_cfg = None
for vh, vd in [(V_HIDDEN_TARGET, V_DEPTH_TARGET), (V_HIDDEN_FALLBACK, V_DEPTH_FALLBACK)]:
    try:
        model, model_cfg = try_model(vh, vd)
        break
    except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
        if 'out of memory' in str(e).lower():
            print(f'  OOM at v_hidden={vh} v_depth={vd} — trying fallback...')
            torch.cuda.empty_cache()
            gc.collect()
        else:
            raise

if model is None:
    raise RuntimeError('Could not fit model even at fallback config')

# ── Auto batch size ──
GRAD_ACCUM = 1
if DEVICE == 'cuda':
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    if vram >= 70:
        BATCH_SIZE = 8
    elif vram >= 40:
        BATCH_SIZE = 4
        GRAD_ACCUM = 2
    else:
        BATCH_SIZE = 2
        GRAD_ACCUM = 4
    # OOM probe at target batch size
    try:
        _rng = np.random.default_rng(42)
        _xb, _yb = get_batch(train_ids, BATCH_SIZE, BLOCK_SIZE, _rng)
        _x = torch.from_numpy(_xb).to(DEVICE)
        _y = torch.from_numpy(_yb).to(DEVICE)
        _, _loss = model(_x, _y)
        _loss.backward()
        model.zero_grad(set_to_none=True)
        del _x, _y, _xb, _yb, _loss
        torch.cuda.empty_cache()
        print(f'OOM probe passed at batch_size={BATCH_SIZE}')
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        model.zero_grad(set_to_none=True)
        gc.collect()
        old_bs = BATCH_SIZE
        BATCH_SIZE = max(BATCH_SIZE // 2, 1)
        GRAD_ACCUM = max(old_bs * GRAD_ACCUM // BATCH_SIZE, 2)
        print(f'OOM at batch_size={old_bs} — falling back to '
              f'batch_size={BATCH_SIZE} x {GRAD_ACCUM} grad accum')
else:
    BATCH_SIZE = 2
    GRAD_ACCUM = 2

EFFECTIVE_BATCH = BATCH_SIZE * GRAD_ACCUM
n_params = model.num_params()
alpha_init_str = ','.join(f'{a:.3f}' for a in model.xi_alpha_values())

print(f'\nModel: FockMultiXiPARFLM v2.1 (OpenWebText Scale-Up)')
print(f'  params: {n_params:,}')
print(f'  d={model_cfg.d}  L={model_cfg.L}  v_hidden={model_cfg.v_hidden}  v_depth={model_cfg.v_depth}')
print(f'  xi_channels={model_cfg.xi_channels}  alpha_init=[{alpha_init_str}]')
print(f'  v_phi_kind={model_cfg.v_phi_kind}  H={model_cfg.v_phi_phi_hidden}  top_k={model_cfg.top_k}')
print(f'  fock_version={model_cfg.fock_version}  M={model_cfg.n_registers}  d_k={model_cfg.d_k}')
print(f'  stack_discipline={model_cfg.stack_discipline}  reverse_channel={model_cfg.reverse_channel}')
print(f'  B1={model_cfg.per_register_tau}  B2={model_cfg.per_register_keys}  B3={model_cfg.ortho_register_init}')
print(f'  fixed_gamma={model_cfg.fixed_gamma}  mass_mode={model_cfg.mass_mode}')
print(f'  gathered_v_phi={model_cfg.use_gathered_v_phi}  layer_ckpt={model_cfg.use_layer_checkpoint}')
print(f'  batch={BATCH_SIZE} x accum={GRAD_ACCUM} (eff={EFFECTIVE_BATCH})')

In [ ]:
# ── Cell 6: Training loop (100k steps, checkpoint resume, Fock grad clip) ─
torch.manual_seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

def lr_schedule(step):
    if step < WARMUP_STEPS:
        return LR * (step + 1) / WARMUP_STEPS
    progress = (step - WARMUP_STEPS) / max(TOTAL_STEPS - WARMUP_STEPS, 1)
    return LR * 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))

def gumbel_tau_schedule(step):
    anneal_start = int(0.2 * TOTAL_STEPS)
    if step < anneal_start:
        return GUMBEL_TAU_INIT
    progress = (step - anneal_start) / max(TOTAL_STEPS - anneal_start, 1)
    progress = min(progress, 1.0)
    return GUMBEL_TAU_INIT + (GUMBEL_TAU_MIN - GUMBEL_TAU_INIT) * progress

@torch.no_grad()
def evaluate():
    model.eval()
    losses = []
    for _ in range(EVAL_ITERS):
        xb, yb = get_batch(val_ids, BATCH_SIZE, BLOCK_SIZE, rng)
        x = torch.from_numpy(xb).to(DEVICE)
        y = torch.from_numpy(yb).to(DEVICE)
        with torch.enable_grad():
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))

def save_checkpoint(step_num, val_loss_val, tag_suffix=''):
    ckpt = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optim.state_dict(),
        'model_cfg': asdict(model_cfg),
        'train_cfg': {
            'batch_size': BATCH_SIZE, 'block_size': BLOCK_SIZE,
            'grad_accum': GRAD_ACCUM, 'effective_batch': EFFECTIVE_BATCH,
            'steps': TOTAL_STEPS, 'lr': LR, 'weight_decay': WEIGHT_DECAY,
            'warmup_steps': WARMUP_STEPS, 'grad_clip': GRAD_CLIP,
            'fock_grad_clip': FOCK_GRAD_CLIP,
        },
        'step': step_num,
        'val_loss': val_loss_val,
        'val_ppl': math.exp(val_loss_val),
        'gamma': model.gamma.item(),
        'xi_alphas': model.xi_alpha_values(),
        'gumbel_tau': model.gumbel_tau(),
        'variant': 'fock_multixi_parf_v21',
        'experiment': f'openwebtext_fock_multixi_scaleup_d256_L12_vh{model_cfg.v_hidden}_vd{model_cfg.v_depth}',
        'corpus': 'openwebtext',
        'fock_version': model_cfg.fock_version,
        'n_registers': model_cfg.n_registers,
        'stack_discipline': model_cfg.stack_discipline,
        'reverse_channel': model_cfg.reverse_channel,
        'seed': SEED,
    }
    fname = f'{CKPT_PREFIX}_step{step_num}{tag_suffix}.pt'
    path = CKPT_DIR / fname
    torch.save(ckpt, path)
    print(f'  Checkpoint saved: {path}')
    return path

# ── Identify Fock-specific parameters for secondary grad clip ──
fock_param_names = set()
for name, _ in model.named_parameters():
    if any(k in name for k in (
        'register_embed', 'creation_gate', 'destruction_gate',
        'reverse_ch', 'reverse_channel_scale',
    )):
        fock_param_names.add(name)
fock_params = [p for n, p in model.named_parameters() if n in fock_param_names]
non_fock_params = [p for n, p in model.named_parameters() if n not in fock_param_names]
print(f'Fock params: {len(fock_params)} tensors  '
      f'({sum(p.numel() for p in fock_params):,} params)')
print(f'Non-Fock params: {len(non_fock_params)} tensors  '
      f'({sum(p.numel() for p in non_fock_params):,} params)')

# ── Optimizer ──
optim = torch.optim.AdamW(
    model.parameters(), lr=LR,
    weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95),
)

# ── Resume from checkpoint ──
if resume_ckpt is not None and resume_step < TOTAL_STEPS:
    print(f'Resuming from checkpoint at step {resume_step}: {resume_ckpt}')
    ckpt_data = torch.load(resume_ckpt, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt_data['model_state_dict'])
    if 'optimizer_state_dict' in ckpt_data:
        optim.load_state_dict(ckpt_data['optimizer_state_dict'])
        print(f'  Optimizer state restored.')
    if 'gumbel_tau' in ckpt_data:
        model.set_gumbel_tau(ckpt_data['gumbel_tau'])
        print(f'  Gumbel tau restored: {ckpt_data["gumbel_tau"]:.4f}')
    print(f'  Model loaded. Starting from step {resume_step + 1}.')
    del ckpt_data
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

# ── Training log ──
log_path = RESULTS_DIR / 'training_log.jsonl'
log_f = log_path.open('a')
loss_history = []

t0 = time.time()
model.train()
running = 0.0
n_run = 0

print(f'\n{"="*60}')
print(f'Fock-PARFLM v2.1 OpenWebText Scale-Up: steps {resume_step+1} -> {TOTAL_STEPS}')
print(f'  batch={BATCH_SIZE} x accum={GRAD_ACCUM} (eff={EFFECTIVE_BATCH})')
print(f'  block={BLOCK_SIZE}  lr={LR}  warmup={WARMUP_STEPS}')
print(f'  v_hidden={model_cfg.v_hidden}  v_depth={model_cfg.v_depth}  params={n_params:,}')
print(f'  K={model_cfg.xi_channels}  H={model_cfg.v_phi_phi_hidden}  top_k={model_cfg.top_k}  M={model_cfg.n_registers}')
print(f'  grad_clip={GRAD_CLIP}  fock_grad_clip={FOCK_GRAD_CLIP}')
print(f'  gumbel_tau: {GUMBEL_TAU_INIT} -> {GUMBEL_TAU_MIN} (anneal last 80%)')
print(f'  checkpoints at steps: {sorted(CKPT_STEPS)}')
print(f'{"="*60}\n')

ckpt_steps_set = set(CKPT_STEPS)

for step in range(resume_step, TOTAL_STEPS):
    # ── LR schedule ──
    lr_now = lr_schedule(step)
    for g in optim.param_groups:
        g['lr'] = lr_now

    # ── Gumbel tau schedule ──
    tau_now = gumbel_tau_schedule(step)
    model.set_gumbel_tau(tau_now)

    # ── Gradient accumulation loop ──
    accum_loss = 0.0
    for _acc in range(GRAD_ACCUM):
        xb, yb = get_batch(train_ids, BATCH_SIZE, BLOCK_SIZE, rng)
        x = torch.from_numpy(xb).to(DEVICE)
        y = torch.from_numpy(yb).to(DEVICE)
        _, loss = model(x, y)
        loss = loss / GRAD_ACCUM
        loss.backward()
        accum_loss += loss.item()

    # ── Two-stage gradient clipping ──
    grad_norm = nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    if fock_params:
        fock_grad_norm = nn.utils.clip_grad_norm_(fock_params, FOCK_GRAD_CLIP)
    else:
        fock_grad_norm = torch.tensor(0.0)

    optim.step()
    optim.zero_grad(set_to_none=True)

    running += accum_loss
    n_run += 1

    if (step + 1) % LOG_INTERVAL == 0:
        avg = running / n_run
        running, n_run = 0.0, 0
        elapsed = time.time() - t0
        gamma_val = model.gamma.item()
        alphas = model.xi_alpha_values()
        alpha_str = ','.join(f'{a:.3f}' for a in alphas)
        print(
            f'step {step+1:6d}/{TOTAL_STEPS}  '
            f'train={avg:.4f}  lr={lr_now:.2e}  '
            f'grad={float(grad_norm):.2f}/{float(fock_grad_norm):.2f}  '
            f'gamma={gamma_val:.3f}  tau={tau_now:.3f}  alpha=[{alpha_str}]  '
            f'{elapsed:.0f}s'
        )
        log_f.write(json.dumps({
            'step': step + 1, 'train_loss': avg,
            'lr': lr_now, 'grad_norm': float(grad_norm),
            'fock_grad_norm': float(fock_grad_norm),
            'gamma': gamma_val, 'gumbel_tau': tau_now,
            'xi_alphas': alphas, 'elapsed_sec': elapsed,
        }) + '\n')
        log_f.flush()

    if (step + 1) % EVAL_INTERVAL == 0:
        vl = evaluate()
        vppl = math.exp(vl)
        elapsed = time.time() - t0
        print(f'  >>> EVAL step {step+1}  val_loss={vl:.4f}  val_ppl={vppl:.2f}  ({elapsed:.0f}s)')
        loss_history.append((step + 1, running / max(n_run, 1) if n_run > 0 else avg, vl))
        log_f.write(json.dumps({
            'step': step + 1, 'val_loss': vl, 'val_ppl': vppl,
        }) + '\n')
        log_f.flush()

    if (step + 1) in ckpt_steps_set:
        vl = evaluate()
        save_checkpoint(step + 1, vl)
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()

log_f.close()
print(f'\nTraining complete ({time.time() - t0:.0f}s total).')

In [ ]:
# ── Cell 7: Final evaluation + summary ────────────────────────────
final_val = evaluate()
final_ppl = math.exp(final_val)
final_gamma = model.gamma.item()
final_alphas = model.xi_alpha_values()
final_tau = model.gumbel_tau()
total_elapsed = time.time() - t0

print(f'\n{"="*60}')
print(f'FOCK-PARFLM v2.1 FINAL  val_loss={final_val:.4f}  val_ppl={final_ppl:.2f}')
print(f'  v_hidden={model_cfg.v_hidden}  v_depth={model_cfg.v_depth}  params={n_params:,}')
print(f'  gamma={final_gamma:.4f}  gumbel_tau={final_tau:.4f}')
print(f'  alpha_final={final_alphas}')
print(f'  elapsed={total_elapsed:.0f}s ({total_elapsed/3600:.2f}h)')
print(f'{"="*60}')

save_checkpoint(TOTAL_STEPS, final_val, tag_suffix='_final')

# ── Loss curve plot ──
if loss_history:
    steps_v, train_vs, val_vs = zip(*loss_history)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(steps_v, val_vs, 'o-', color='darkorange', label='val loss')
    ax1.set_xlabel('step')
    ax1.set_ylabel('loss (nats)')
    ax1.set_title('Validation loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(steps_v, [math.exp(v) for v in val_vs], 'o-', color='royalblue', label='val ppl')
    ax2.set_xlabel('step')
    ax2.set_ylabel('perplexity')
    ax2.set_title('Validation perplexity')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    fig.suptitle(
        f'Fock-PARFLM v2.1 OpenWebText Scale-Up\n'
        f'd={model_cfg.d}  L={model_cfg.L}  v_hidden={model_cfg.v_hidden}  '
        f'v_depth={model_cfg.v_depth}  K={model_cfg.xi_channels}  '
        f'M={model_cfg.n_registers}  params={n_params:,}',
        fontsize=11,
    )
    fig.tight_layout()
    fig_path = RESULTS_DIR / 'training_curve.png'
    fig.savefig(fig_path, dpi=150)
    print(f'Loss curve saved: {fig_path}')
    plt.show()

# ── Xi-alpha evolution plot ──
try:
    import json as _json
    log_entries = []
    with open(RESULTS_DIR / 'training_log.jsonl') as f:
        for line in f:
            e = _json.loads(line)
            if 'xi_alphas' in e:
                log_entries.append(e)
    if log_entries:
        K = len(log_entries[0]['xi_alphas'])
        fig2, ax = plt.subplots(figsize=(10, 5))
        for k in range(K):
            ax.plot([e['step'] for e in log_entries],
                    [e['xi_alphas'][k] for e in log_entries],
                    label=f'alpha_{k}')
        ax.set_xlabel('step')
        ax.set_ylabel('alpha_k')
        ax.set_title('Xi-channel alpha evolution')
        ax.legend()
        ax.grid(True, alpha=0.3)
        fig2.tight_layout()
        fig2_path = RESULTS_DIR / 'xi_alpha_evolution.png'
        fig2.savefig(fig2_path, dpi=150)
        print(f'Alpha evolution plot saved: {fig2_path}')
        plt.show()
except Exception as e:
    print(f'Could not plot alpha evolution: {e}')

# ── Summary markdown ──
summary_path = RESULTS_DIR / 'training_summary.md'
with summary_path.open('w') as f:
    f.write('# Training summary — Fock-PARFLM v2.1 OpenWebText Scale-Up\n\n')
    f.write('- model: FockMultiXiPARFLM (v2.1 = v2 + B1 + B2 + B3)\n')
    f.write(f'- corpus: OpenWebText (~{MAX_TRAIN_TOKENS//1_000_000}M train tokens)\n')
    f.write(f'- params: {n_params:,}\n')
    f.write(f'- d={model_cfg.d}  L={model_cfg.L}  v_hidden={model_cfg.v_hidden}  v_depth={model_cfg.v_depth}\n')
    f.write(f'- xi_channels={model_cfg.xi_channels}  alpha_final={final_alphas}\n')
    f.write(f'- v_phi_kind={model_cfg.v_phi_kind}  H={model_cfg.v_phi_phi_hidden}  top_k={model_cfg.top_k}\n')
    f.write(f'- fock_version={model_cfg.fock_version}  M={model_cfg.n_registers}  d_k={model_cfg.d_k}\n')
    f.write(f'- stack_discipline={model_cfg.stack_discipline}  reverse_channel={model_cfg.reverse_channel}\n')
    f.write(f'- B1(per_register_tau)={model_cfg.per_register_tau}  '
            f'B2(per_register_keys)={model_cfg.per_register_keys}  '
            f'B3(ortho_init)={model_cfg.ortho_register_init}\n')
    f.write(f'- fixed_gamma: {model_cfg.fixed_gamma}\n')
    f.write(f'- gumbel_tau_final: {final_tau:.4f}  (init={GUMBEL_TAU_INIT} min={GUMBEL_TAU_MIN})\n')
    f.write(f'- grad_clip: {GRAD_CLIP}  fock_grad_clip: {FOCK_GRAD_CLIP}\n')
    f.write(f'- batch_size={BATCH_SIZE}  block_size={BLOCK_SIZE}  '
            f'grad_accum={GRAD_ACCUM}  steps={TOTAL_STEPS}\n')
    f.write(f'- seed: {SEED}\n')
    f.write(f'- elapsed: {total_elapsed:.0f}s ({total_elapsed/3600:.2f}h)\n\n')
    f.write(f'Final val loss: {final_val:.6f} (ppl {final_ppl:.2f})\n')
    f.write(f'Final gamma: {final_gamma:.4f}\n')
    f.write(f'Final alpha: {final_alphas}\n')
    f.write(f'Final gumbel_tau: {final_tau:.4f}\n\n')
    f.write('## Three-Branch Comparison\n\n')
    f.write('| Branch | OWT val PPL | Params | Steps |\n')
    f.write('|--------|------------|--------|-------|\n')
    f.write(f'| Multi-Xi SPLM (Phase 3) | 153.89 | 28.3M | 100k |\n')
    f.write(f'| Multi-Xi PARFLM | -- | -- | 100k |\n')
    f.write(f'| **Fock-PARFLM v2.1** | **{final_ppl:.2f}** | **{n_params/1e6:.1f}M** | **{TOTAL_STEPS//1000}k** |\n')
print(f'Summary: {summary_path}')

In [ ]:
# ── Cell 8: (Optional) Push checkpoint to HuggingFace Hub ─────────
PUSH_TO_HF = False
HF_REPO     = 'dimitarpg13/semsimula-fock-parflm-owt-scaleup'
HF_TOKEN    = ''

if PUSH_TO_HF:
    from huggingface_hub import HfApi, create_repo
    token = HF_TOKEN or os.environ.get('HF_TOKEN', '')
    if not token:
        print('No HF token — skipping push. Set HF_TOKEN env var or paste above.')
    else:
        api = HfApi(token=token)
        try:
            create_repo(HF_REPO, repo_type='model', exist_ok=True, token=token)
        except Exception as e:
            print(f'  (repo may already exist: {e})')

        final_ckpt = CKPT_DIR / f'{CKPT_PREFIX}_step{TOTAL_STEPS}_final.pt'
        if final_ckpt.exists():
            print(f'Uploading {final_ckpt} to {HF_REPO}...')
            api.upload_file(
                path_or_fileobj=str(final_ckpt),
                path_in_repo=final_ckpt.name,
                repo_id=HF_REPO,
                token=token,
            )
            print('Upload complete.')
        else:
            print(f'Final checkpoint not found at {final_ckpt}')
else:
    print('HF push disabled. Set PUSH_TO_HF = True to upload.')